In [2]:
import torch

In [3]:
x = torch.arange(12)
x

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [4]:
x.shape

torch.Size([12])

In [5]:
x.numel()

12

`reshape`：改变一个张量的形状，但不改变元素数量和元素值

In [6]:
X = x.reshape(3, 4)
X

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [7]:
torch.zeros((2, 3, 4))

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [8]:
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
Y = torch.tensor([[2.0,1,4,3], [1,2,3,4], [4,3,2,1]])
torch.cat((X, Y), dim=0),torch.cat((X, Y), dim=1) #两个元素在第0维(行方向)和第1维（列方向）上连接

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

In [9]:
X.sum()

tensor(66.)

In [10]:
X[-1],X[1:3]

(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

创建一个人工数据集，并存储在csv（逗号分隔值）文件

In [12]:
import os
os.makedirs('data', exist_ok=True)
data_file = os.path.join('data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n') #写入表头
    f.write('NA,Pave,127500\n') #写入第一行数据
    f.write('2,NA,106000\n') #写入第二行数据
    f.write('4,NA,178100\n') #写入第三行数据
    f.write('NA,NA,140000\n') #写入第四行数据

`data_file` 完整文件路径data/house_tiny.csv

`with` 上下文管理器，确保文件在代码块执行完毕后自动关闭，如果不用with则需在末尾手动加上f.close()。

该段代码运行后可在当前目录下看到新增文件夹data

**从创建的csv文件中加载原始数据集**

一般使用pandas读取csv文件

In [13]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


为了处理缺失的数据，典型方法为插值，删除。这里使用插值

In [20]:
inputs, outputs = data.iloc[:,0:2],data.iloc[:,2]
inputs = inputs.fillna(inputs.mean(numeric_only=True)) #用均值填充缺失值
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


`inputs.mean()` 计算 inputs 中每一列的均值。

`fillna()` 将 inputs 中的缺失值（NaN）替换为对应列的均值。

In [21]:
inputs = pd.get_dummies(inputs, dummy_na=True,dtype=int) #将NAN值视为一个类别
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0           1          0
1       2.0           0          1
2       4.0           0          1
3       3.0           0          1


默认情况下，`pd.get_dummies` 会忽略缺失值（NaN），不为其创建列，并且在生成的哑变量中，缺失值对应的所有哑变量都是 0。

设置 `dummy_na=True` 后，会额外为缺失值创建一个专门的列（例如 Alley_nan），将缺失值视为一个独立的类别。

In [22]:
import torch

X, y = torch.tensor(inputs.values), torch.tensor(outputs.values)
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500, 106000, 178100, 140000]))